# Road Extraction

`Runtime` -> `Run all`. It asks for coordinates, then shows what each model found.

Use a GPU if you can: `Runtime` -> `Change runtime type` -> `T4 GPU`.

### 1. Setup

In [ ]:
%cd /content
!rm -rf CNN && git clone -q --depth 1 https://github.com/fsRakib/Road-Extraction.git CNN
%cd /content/CNN
!mkdir -p models/weights data/images data/osm outputs/images outputs/geojson reports
!pip -q install rasterio shapely pyproj scikit-image segmentation-models-pytorch

# pretrained weights
!wget -q "https://www.dropbox.com/sh/h62vr320eiy57tt/AAB5Tm43-efmtYzW_GFyUCfma?dl=1" -O /tmp/d.zip
!unzip -oq /tmp/d.zip -d /tmp/d && cp /tmp/d/log01_dink34.th models/weights/dlinknet34_deepglobe.th
!wget -q "https://huggingface.co/teohyc/Satellite-Road-Segmentation-UNet/resolve/main/best_road_seg_unet.pth" \
      -O models/weights/unet_road_massachusetts.pth

from pathlib import Path
for f in ["dlinknet34_deepglobe.th", "unet_road_massachusetts.pth"]:
    p = Path("models/weights") / f
    mb = p.stat().st_size / 1e6 if p.exists() else 0
    print(f"{'ok     ' if mb > 1 else 'MISSING'}  {f}  {mb:.0f} MB")

### 2. Coordinates

In [ ]:
import re
from IPython.display import Image, display

lat, lon = (float(t) for t in re.split(r"[,\s]+", input("Coordinates (lat, lon): ").strip())[:2])
name = re.sub(r"[^0-9A-Za-z._-]", "_", input("Name (blank = auto): ").strip()) or f"aoi_{lat:.4f}_{lon:.4f}"

!python download.py {lat} {lon} {name}
display(Image(f"data/images/{name}.png", width=450))

### 3. Results

In [ ]:
MODELS = ["baseline", "dlinknet", "unet"]

import json, time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image as PILImage
import extract
from core.registry import discover

model_label = {n: cls.description for n, cls in discover().items()}

stats = {}
for m in MODELS:
    t0 = time.time()
    try:
        extract.run(name, m)
        gj = json.loads(Path(f"outputs/geojson/{name}_{m}.geojson").read_text())
        km = sum(f["properties"]["length_m"] for f in gj["features"]) / 1000
        stats[m] = (len(gj["features"]), km, time.time() - t0)
    except (Exception, SystemExit) as e:
        print(f"\n*** {m} FAILED: {type(e).__name__}: {e}\n")

done = [m for m in MODELS if m in stats]

# top row = red overlay on the satellite image, bottom row = the raw mask
fig, axes = plt.subplots(2, len(done) + 1, figsize=(5.5 * (len(done) + 1), 11))
axes = np.array(axes).reshape(2, -1)

axes[0][0].imshow(PILImage.open(f"data/images/{name}.png"))
axes[0][0].set_title("satellite", fontsize=13)
axes[1][0].axis("off")

for col, m in enumerate(done, start=1):
    lines, km, secs = stats[m]
    label = model_label[m]
    axes[0][col].imshow(PILImage.open(f"outputs/images/{name}_{m}.png"))
    axes[0][col].set_title(f"{label}\n{lines} lines | {km:.1f} km | {secs:.0f}s", fontsize=13)
    axes[1][col].imshow(PILImage.open(f"outputs/images/{name}_{m}_mask.png"), cmap="gray")
    axes[1][col].set_title(f"{label} - mask", fontsize=13)

for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()


### 4. Download

In [ ]:
from google.colab import files
for m in done:
    files.download(f"outputs/geojson/{name}_{m}.geojson")